In [ ]:
#!/usr/bin/env python3
"""
Sentinel-2 NDVI - WASP Yearly Mosaic Creation

WORKFLOW:
1. For each year:
   - Process all tiles (NDVI median May-Sep)
   - Merge all tiles into a Bavaria mosaic
   - Save only the mosaic (0-100 scaling)
   - Delete individual tiles

Author: Agnes Zwick
Date: February 2026
Version: 5.0 - Yearly Mosaic
"""

from pystac_client import Client
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.merge import merge
import numpy as np
from pathlib import Path
from datetime import datetime
import time
import shutil

# ============================================================================
# CONFIGURATION
# ============================================================================

BASE_OUTPUT_DIR = Path('/dss/dsshome1/03/di97bag/Masterarbeit/02_Daten/Output/Sentinel2_NDVI_Bayern_Yearly')
TEMP_DIR = BASE_OUTPUT_DIR / 'temp'
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(exist_ok=True)

STAC_URL = "https://geoservice.dlr.de/eoc/ogc/stac/v1"
COLLECTION = "S2_L3A_WASP"

# Bavaria tiles
TILES = [
    "32TNT", "32TPT", "32TQT",
    "32UMA", "32UMV", "32UNA", "32UNB", "32UNU", "32UNV",
    "32UPA", "32UPB", "32UPU", "32UPV",
    "32UQA", "32UQU", "32UQV",
    "33TUN", "33UUP", "33UUQ", "33UUR", "33UVP", "33UVQ",
]

YEARS = range(2025, 2026)
SUMMER_MONTHS = [5, 6, 7, 8, 9]

MAX_RETRIES = 3
RETRY_DELAY = 2

# ============================================================================
# UTILITY
# ============================================================================

def log(message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] {message}")

def mgrs_to_stac_format(tile):
    """32UMA → 32/U/MA"""
    zone = tile[:2]
    square = tile[2]
    grid = tile[3:5]
    return f"{zone}/{square}/{grid}"

# ============================================================================
# STAC SEARCH
# ============================================================================

def get_monthly_synthesis(tile, year, month):
    """Fetch ONE monthly synthesis for a tile/year/month"""
    
    try:
        catalog = Client.open(STAC_URL)
    except Exception as e:
        log(f"      STAC error: {e}")
        return None
    
    start_date = f"{year}-{month:02d}-01"
    if month == 12:
        end_date = f"{year+1}-01-01"
    else:
        end_date = f"{year}-{month+1:02d}-01"
    
    stac_tile = mgrs_to_stac_format(tile)
    
    try:
        search = catalog.search(
            collections=[COLLECTION],
            datetime=f"{start_date}/{end_date}",
            query={
                "grid:code": {"eq": stac_tile}
            },
            limit=10
        )
        
        items = list(search.items())
        
        if not items:
            return None
        
        item = items[0]
        
        if 'FRC_B4' not in item.assets or 'FRC_B8' not in item.assets:
            log(f"    ⚠ Missing bands in item")
            return None
        
        return item
        
    except Exception as e:
        log(f"      Search error: {e}")
        return None

# ============================================================================
# NDVI CALCULATION
# ============================================================================

def compute_ndvi_from_synthesis(item, reference_extent=None):
    """Compute NDVI from monthly synthesis"""
    
    red_asset = item.assets['FRC_B4']
    nir_asset = item.assets['FRC_B8']
    
    for attempt in range(MAX_RETRIES):
        try:
            if reference_extent:
                height, width = reference_extent['shape']
                red_data = np.zeros((height, width), dtype=np.float32)
                nir_data = np.zeros((height, width), dtype=np.float32)
                
                with rasterio.open(red_asset.href) as src:
                    reproject(
                        source=rasterio.band(src, 1),
                        destination=red_data,
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=reference_extent['transform'],
                        dst_crs=reference_extent['crs'],
                        resampling=Resampling.bilinear
                    )
                
                with rasterio.open(nir_asset.href) as src:
                    reproject(
                        source=rasterio.band(src, 1),
                        destination=nir_data,
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=reference_extent['transform'],
                        dst_crs=reference_extent['crs'],
                        resampling=Resampling.bilinear
                    )
                
                profile = {
                    'driver': 'GTiff',
                    'dtype': rasterio.float32,
                    'nodata': np.nan,
                    'width': width,
                    'height': height,
                    'count': 1,
                    'crs': reference_extent['crs'],
                    'transform': reference_extent['transform'],
                    'compress': 'lzw'
                }
                
            else:
                with rasterio.open(red_asset.href) as src:
                    red_data = src.read(1).astype(float)
                    profile = src.profile.copy()
                    profile.update(dtype=rasterio.float32, nodata=np.nan)
                
                with rasterio.open(nir_asset.href) as src:
                    nir_data = src.read(1).astype(float)
            
            # NDVI calculation
            red_refl = red_data / 10000.0
            nir_refl = nir_data / 10000.0
            
            denominator = nir_refl + red_refl
            mask = denominator > 0
            
            ndvi = np.full_like(red_data, np.nan, dtype=np.float32)
            ndvi[mask] = (nir_refl[mask] - red_refl[mask]) / denominator[mask]
            
            ndvi[(ndvi < -1) | (ndvi > 1)] = np.nan
            
            return ndvi, profile
            
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                log(f"    Retry {attempt+1}/{MAX_RETRIES}: {e}")
                time.sleep(RETRY_DELAY)
            else:
                log(f"      Error: {e}")
                return None, None
    
    return None, None

def get_reference_extent(item):
    """Get extent from first synthesis"""
    try:
        red_asset = item.assets['FRC_B4']
        with rasterio.open(red_asset.href) as src:
            return {
                'bounds': src.bounds,
                'crs': src.crs,
                'transform': src.transform,
                'width': src.width,
                'height': src.height,
                'shape': (src.height, src.width)
            }
    except Exception as e:
        log(f"      Extent error: {e}")
        return None

# ============================================================================
# TILE PROCESSING
# ============================================================================

def process_tile_for_year(tile, year, temp_dir):
    """
    Process one tile for one year
    Creates MEDIAN NDVI over summer months
    Returns path to the temporary tile file
    """
    
    log(f"  └─ Tile {tile}...")
    
    # 1. Get monthly syntheses
    monthly_items = []
    for month in SUMMER_MONTHS:
        item = get_monthly_synthesis(tile, year, month)
        if item:
            monthly_items.append((month, item))
    
    if not monthly_items:
        log(f"       No syntheses found")
        return None
    
    log(f"     ✓ {len(monthly_items)}/5 months available")
    
    # 2. Determine extent
    reference_extent = get_reference_extent(monthly_items[0][1])
    if not reference_extent:
        log(f"       Could not determine extent")
        return None
    
    # 3. Compute NDVI for each month
    ndvi_arrays = []
    profile = None
    
    for month, item in monthly_items:
        ndvi, prof = compute_ndvi_from_synthesis(item, reference_extent)
        
        if ndvi is not None:
            ndvi_arrays.append(ndvi)
            if profile is None:
                profile = prof
    
    if not ndvi_arrays:
        log(f"       No successful NDVI calculations")
        return None
    
    # 4. Compute MEDIAN
    ndvi_stack = np.stack(ndvi_arrays, axis=0)
    ndvi_median = np.nanmedian(ndvi_stack, axis=0).astype(np.float32)
    
    # 5. Scale to 0-100 and convert to Int16
    # NDVI: -1 to 1 → scaled: 0 to 100
    # Formula: (ndvi + 1) * 50 = 0-100
    ndvi_scaled = ((ndvi_median + 1) * 50).astype(np.int16)
    
    # NoData as -9999
    ndvi_scaled[np.isnan(ndvi_median)] = -9999
    
    # 6. Update profile for Int16
    profile.update(
        dtype=rasterio.int16,
        nodata=-9999
    )
    
    # 7. Save temporarily
    temp_file = temp_dir / f"NDVI_{tile}_{year}_temp.tif"
    with rasterio.open(temp_file, 'w', **profile) as dst:
        dst.write(ndvi_scaled, 1)
    
    valid_count = np.sum(ndvi_scaled != -9999)
    median_val = np.nanmedian(ndvi_median)
    log(f"     ✓ NDVI Median: {median_val:.3f}, Valid: {valid_count:,} px")
    
    # Cleanup
    del ndvi_arrays, ndvi_stack, ndvi_median, ndvi_scaled
    
    return temp_file

# ============================================================================
# MOSAIC CREATION
# ============================================================================

def create_mosaic(tile_files, year, output_dir):
    """
    Create mosaic from all tiles
    Save as Bayern_NDVI_YYYY.tif
    """
    
    log(f"\n  Creating mosaic from {len(tile_files)} tiles...")
    
    # Open all tiles
    src_files = []
    for tile_file in tile_files:
        src = rasterio.open(tile_file)
        src_files.append(src)
    
    # Merge
    mosaic, out_trans = merge(src_files, nodata=-9999)
    
    # Profile from first tile
    out_meta = src_files[0].meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_trans,
        "compress": "lzw",
        "dtype": rasterio.int16,
        "nodata": -9999
    })
    
    # Save mosaic
    mosaic_file = output_dir / f"Bayern_NDVI_{year}_summer_median.tif"
    with rasterio.open(mosaic_file, "w", **out_meta) as dest:
        dest.write(mosaic)
    
    # Close all sources
    for src in src_files:
        src.close()
    
    # Statistics
    valid_pixels = np.sum(mosaic != -9999)
    total_pixels = mosaic.size
    coverage = (valid_pixels / total_pixels) * 100
    
    log(f"  ✓ Mosaic created: {mosaic_file.name}")
    log(f"    Size: {mosaic.shape[2]} x {mosaic.shape[1]} px")
    log(f"    Valid: {valid_pixels:,} px ({coverage:.1f}%)")
    log(f"    Range: {np.min(mosaic[mosaic != -9999])} - {np.max(mosaic[mosaic != -9999])}")
    
    return mosaic_file

# ============================================================================
# YEAR PROCESSING
# ============================================================================

def process_year(year, base_output_dir, temp_dir):
    """
    Process one complete year
    1. All tiles → compute NDVI median
    2. Create mosaic
    3. Delete temp files
    """
    
    log(f"\n{'='*80}")
    log(f"YEAR {year}")
    log(f"{'='*80}")
    
    # Check if mosaic already exists
    mosaic_file = base_output_dir / f"Bayern_NDVI_{year}_summer_median.tif"
    if mosaic_file.exists():
        log(f"✓ Mosaic already exists: {mosaic_file.name}")
        return True
    
    # 1. Process all tiles
    log(f"\nProcessing {len(TILES)} tiles...")
    tile_files = []
    
    for i, tile in enumerate(TILES, 1):
        log(f"\n[{i}/{len(TILES)}] {tile}")
        tile_file = process_tile_for_year(tile, year, temp_dir)
        
        if tile_file:
            tile_files.append(tile_file)
    
    if not tile_files:
        log(f"\n No tiles successfully processed for {year}")
        return False
    
    log(f"\n✓ {len(tile_files)}/{len(TILES)} tiles successful")
    
    # 2. Create mosaic
    try:
        mosaic_file = create_mosaic(tile_files, year, base_output_dir)
    except Exception as e:
        log(f"\n Mosaic error: {e}")
        import traceback
        traceback.print_exc()
        return False
    
    # 3. Delete temporary tile files
    log(f"\n  Deleting temporary tiles...")
    for tile_file in tile_files:
        try:
            tile_file.unlink()
        except Exception as e:
            log(f"Could not delete {tile_file.name}: {e}")
    
    log(f"  ✓ Cleanup complete")
    
    log(f"\n{'='*80}")
    log(f"✓ YEAR {year} COMPLETE")
    log(f"{'='*80}")
    
    return True

# ============================================================================
# MAIN
# ============================================================================

def main():
    """Main function - process all years"""
    
    log("="*80)
    log("SENTINEL-2 NDVI - YEARLY BAVARIA MOSAICS")
    log("="*80)
    log(f"Tiles:        {len(TILES)}")
    log(f"Years:        {min(YEARS)}-{max(YEARS)} ({len(YEARS)} years)")
    log(f"Months:       {SUMMER_MONTHS} (monthly syntheses)")
    log(f"Scaling:      0-100 (Int16)")
    log(f"Output:       {BASE_OUTPUT_DIR}")
    log(f"Temp dir:     {TEMP_DIR}")
    log("="*80)
    
    start_time = datetime.now()
    results = {'success': 0, 'failed': 0}
    
    # Process year by year
    for year in YEARS:
        try:
            success = process_year(year, BASE_OUTPUT_DIR, TEMP_DIR)
            results['success' if success else 'failed'] += 1
            
        except Exception as e:
            log(f"\nERROR in year {year}: {e}")
            import traceback
            traceback.print_exc()
            results['failed'] += 1
    
    # Cleanup: delete temp_dir if empty
    try:
        if TEMP_DIR.exists() and not any(TEMP_DIR.iterdir()):
            TEMP_DIR.rmdir()
            log(f"\n✓ Temp directory deleted")
    except:
        pass
    
    duration = datetime.now() - start_time
    
    log("\n" + "="*80)
    log("SUMMARY")
    log("="*80)
    log(f"Successful years: {results['success']}/{len(YEARS)}")
    log(f"Failed:            {results['failed']}/{len(YEARS)}")
    log(f"Duration:          {duration}")
    log(f"Output files:      {BASE_OUTPUT_DIR}")
    log("="*80)
    
    # List final files
    mosaic_files = list(BASE_OUTPUT_DIR.glob("Bayern_NDVI_*.tif"))
    if mosaic_files:
        log(f"\nCreated mosaics ({len(mosaic_files)}):")
        for f in sorted(mosaic_files):
            size_mb = f.stat().st_size / (1024**2)
            log(f"  - {f.name} ({size_mb:.1f} MB)")

if __name__ == '__main__':
    main()